## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake

NOTE that the file manager will need to be updated for this workflow to run!!!
Further testing is needed.


### Write snakemake file ###

This block of code writes the snakemake file

In [1]:

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))


from scripts import gpep_to_summa_utils as gts_utils
config = {}
config['summa_forcing_dir'] = Path('/Users/dcasson/Data/gpep/chena/forcing/summa/')

ens, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            if str(file_path.parent) == ens_member:
                f.write(f'{file}.nc\n')
    forcing_file_list.append(forcing_list_file)


print(ens)
print(file_paths)


{'004', '006', '008', '009', '001', '002', '005', '010', '003', '007'}
['007/rf_static_boxcox_low_predictors_ensMember_007', '009/rf_static_boxcox_low_predictors_ensMember_009', '008/rf_static_boxcox_low_predictors_ensMember_008', '001/rf_static_boxcox_low_predictors_ensMember_001', '006/rf_static_boxcox_low_predictors_ensMember_006', '010/rf_static_boxcox_low_predictors_ensMember_010', '003/rf_static_boxcox_low_predictors_ensMember_003', '004/rf_static_boxcox_low_predictors_ensMember_004', '005/rf_static_boxcox_low_predictors_ensMember_005', '002/rf_static_boxcox_low_predictors_ensMember_002']


In [20]:
%%writefile ../rules/run_pysumma_snakemake_test.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user

"""

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils

# UPDATE LOCAL SUMMA PATH
config['summa_exe'] = '/Users/dcasson/GitHub/summa/bin/summa.exe'
config['summa_forcing_dir'] = Path('/Users/dcasson/Data/gpep/chena/forcing/summa/')

# Resolve all file paths and directories in the config file
config['file_manager'] = '/Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt'
config['summa_output_dir'] = '/Users/dcasson/Data/yukon_esp/summa_output/'
config['attributes_nc'] = '/Users/dcasson/Data/yukon_esp/summa/settings/attributes.nc'
config['case_name'] = 'chena'
config['run_suffix'] = 'base'


ens_set, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])
ens = list(ens_set)

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            if str(file_path.parent) == ens_member:
                f.write(f'{file}.nc\n')
    forcing_file_list.append(forcing_list_file)

rule run_summa_base_simulations:
    input:
        expand(Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc"),ens_member=ens)

rule run_summa_ensemble_simulations:
    input:
        file_manager = Path(config['file_manager']),
        forcing_file = lambda wildcards: forcing_file_list[ens.index(wildcards.ens_member)]
    output:
        summa_chunked_output = Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        run_suffix = lambda wildcards: wildcards.ens_member,
    run:
        print(input.forcing_file)
        sim = ps.Simulation(params.summa_exe, input.file_manager)
        sim._force_file_list = Path(input.forcing_file)
        print(sim.force_file_list)
        print(str(params.run_suffix))
        sim.run(run_suffix=str(params.run_suffix))


        

Overwriting ../rules/run_pysumma_snakemake_test.smk


In [ ]:
import pysumma as ps
summa_exe = '/Users/dcasson/GitHub/summa/bin/summa.exe'
file_manager = '/Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt'
forcing_file = '/Users/dcasson/Data/yukon_esp/summa/forcing/summa/forcing_files_001.txt'

sim = ps.Simulation(summa_exe, file_manager)
sim.assign


AttributeError: 'Simulation' object has no attribute 'assign'

In [ ]:
! snakemake --unlock -s ../rules/run_pysumma_snakemake_test.smk #--configfile ../config/gpep_to_summa_tuolumne_config.yaml
! snakemake -s ../rules/run_pysumma_snakemake_test.smk --cores 8 #--configfile ../config/gpep_to_summa_tuolumne_config.yaml


Unlocking working directory.
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 8
Rules claiming more threads will be scaled down.
Job stats:
job                               count    min threads    max threads
------------------------------  -------  -------------  -------------
run_summa_base_simulations            1              1              1
run_summa_ensemble_simulations       10              1              1
total                                11              1              1

Select jobs to execute...

[Mon Nov 18 13:26:57 2024]
rule run_summa_ensemble_simulations:
    input: /Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt, /Users/dcasson/Data/gpep/chena/forcing/summa/forcing_files_010.txt
    output: /Users/dcasson/Data/yukon_esp/summa_output/chena_010_timestep.nc
    jobid: 2
    reason: Missing output files: /Users/dcasson/Data/yukon_esp/summa_output/chena_010_timestep.nc
    wildcards: ens_member=010
    resources: tmpdir=/var/folders/w4/qfhd

In [15]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_simulation

# Resolve all file paths and directories in the config file
config['summa_output_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['working_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['summa_interim_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/interim'
config['fsca_output_dir'] =  '/Users/drc858/Data/gpep/RF_ens/summa/output/fsca'
config_file_path = '/Users/drc858/GitHub/snow_dist/settings/config_summa_model_tuolumne.yaml'

#Read first raw forcing file for easymore remapping
summa_result_filepaths = list(Path(config['summa_output_dir']).glob('*.nc'))
summa_filenames = [Path(filepath).name for filepath in summa_result_filepaths]
print(summa_filenames)

fsca_simulation = summa_simulation.SummaSimulation(config_file_path)

# Prepare summa model results for fsca
# This includes summing the input and output snowpack variables
rule run_fsca_model:
    input:
        expand(Path(config['summa_interim_dir'], "{filenames}"))

rule prepare_fsca_model_input:
    input:
        summa_result = Path(config['summa_output_dir'],"{filenames}")
    output:
        fsca_input = Path(config['summa_interim_dir'],"{filenames}")
    run:
        summa_ds = xr.open_dataset(input.input_summa_result)
        fsca_ds = summa_fsca_model.prepare_summa_fsca_input(summa_ds)
        fsca_ds.to_netcdf(output.fsca_input)

# Run fsca settings
rule run_fsca_simulations:
    input:
        prepped_files = expand(Path(config['summa_interim_dir'], "{filenames}"), filenames=summa_filenames)
    output:
        fsca_result = Path(config['fsca_output_dir'], "{filenames}")
    run:
        sim = summa_simulation.SummaSimulation(config)
        summa_fsca_model.run_summa_fsca_model(input.prepped_files,output.fsca_result)



Overwriting ../rules/run_fsca_simulations.smk


In [16]:
#-s ../rules/run_pysumma_snakemake.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml

ImportError in file /Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk, line 13:
cannot import name 'ss_utils' from 'scripts' (unknown location)
  File "/Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk", line 13, in <module>
